In [ ]:
!pip install --upgrade transformers sentencepiece faiss-cpu -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.8 MB/s eta 0:00:00
PyTorch version: 2.10.0+cpu
CUDA available: False


In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
import faiss

print(f"FAISS version: {faiss.__version__}")

FAISS version: 1.13.2


In [ ]:
DATA_DIR   = ''
OUTPUT_DIR = '/retrieval'

MODEL_NAME = "laberta_vulg"

MAX_K    = 100
K_VALUES = [5, 10, 20, 100]   # kept for eval compatibility

BDC_CSV       = f"{DATA_DIR}/VD_bdc_chunks.csv"
PSC_CSV       = f"{DATA_DIR}/VD_psc_chunks.csv"
TEXT_COLUMN   = "text"
ID_COLUMN     = "seg_id"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
def load_embeddings(embeddings_dir, model_name):

    bdc_embeddings = np.load(f"{embeddings_dir}/bdc_embeddings_{model_name}.npy")
    psc_embeddings = np.load(f"{embeddings_dir}/psc_embeddings_{model_name}.npy")
    print(f"BDC shape: {bdc_embeddings.shape}")
    print(f"PSC shape: {psc_embeddings.shape}")

    with open(f"{embeddings_dir}/bdc_ids_{model_name}.json") as f:
        bdc_ids = json.load(f)
    with open(f"{embeddings_dir}/psc_ids_{model_name}.json") as f:
        psc_ids = json.load(f)
    print(f"BDC IDs: {len(bdc_ids):,}  |  PSC IDs: {len(psc_ids):,}")

    with open(f"{embeddings_dir}/metadata_{model_name}.json") as f:
        metadata = json.load(f)

    assert bdc_embeddings.shape[0] == len(bdc_ids)
    assert psc_embeddings.shape[0] == len(psc_ids)
    assert bdc_embeddings.shape[1] == psc_embeddings.shape[1]
    print("All checks passed")
    return bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata


bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata = load_embeddings(DATA_DIR, MODEL_NAME)

In [ ]:
# Load raw text for reranker
bdc_df = pd.read_csv(BDC_CSV)
psc_df = pd.read_csv(PSC_CSV)

# Build id → text lookup dicts
if ID_COLUMN and ID_COLUMN in bdc_df.columns:
    bdc_text_lookup = dict(zip(bdc_df[ID_COLUMN], bdc_df[TEXT_COLUMN]))
    psc_text_lookup = dict(zip(psc_df[ID_COLUMN], psc_df[TEXT_COLUMN]))
else:
    # Fallback
    bdc_text_lookup = bdc_df[TEXT_COLUMN].to_dict()
    psc_text_lookup = psc_df[TEXT_COLUMN].to_dict()

print(f"BDC texts loaded: {len(bdc_text_lookup):,}")
print(f"PSC texts loaded: {len(psc_text_lookup):,}")

# sanity check
sample_id = bdc_ids[0]
print(f"\nSample BDC id : {sample_id}")
print(f"Sample text   : {str(bdc_text_lookup.get(sample_id, 'NOT FOUND'))[:120]}...")

In [ ]:
# build FAISS index
embedding_dim = psc_embeddings.shape[1]
print(f"Embedding dimension: {embedding_dim}")

index = faiss.IndexFlatIP(embedding_dim)
index.add(psc_embeddings.astype('float32'))

print(f"FAISS IndexFlatIP built")
print(f"Vectors: {index.ntotal:,}")

bdc_id_to_idx = {chunk_id: idx for idx, chunk_id in enumerate(bdc_ids)}

In [ ]:
# retroeve top 100
retrieval_results = {}   # query_id → [(candidate_id, similarity), ...] top-100

print(f"Retrieving top-{MAX_K} candidates for {len(bdc_ids):,} queries...")

for query_id in tqdm(bdc_ids, desc="Retrieving"):
    query_idx = bdc_id_to_idx[query_id]
    distances, indices = index.search(
        bdc_embeddings[query_idx:query_idx+1].astype('float32'),
        MAX_K
    )
    candidates = [
        (psc_ids[idx], float(dist))
        for dist, idx in zip(distances[0], indices[0])
    ]
    retrieval_results[query_id] = candidates

print(f"Retrieval complete")
print(f"  Queries  : {len(retrieval_results):,}")
print(f"  Top-k    : {MAX_K}")

sample_qid = bdc_ids[0]
print(f"\nTop-3 for {sample_qid}:")
for cand_id, score in retrieval_results[sample_qid][:3]:
    print(f"  {score:.4f}  {cand_id}")

Retrieving top-100 candidates for 19,466 queries...


Retrieving:   0%|          | 0/19466 [00:00<?, ?it/s]

Retrieval complete
  Queries  : 19,466
  Top-k    : 100

Top-3 for 10015_sent_0:
  0.5686  066_Benedictus-Nursiae_Regula-cum-commentariis_window_520
  0.5678  066_Benedictus-Nursiae_Regula-cum-commentariis_window_698
  0.5634  031_Augustinus-Hipponensis_Confessiones_window_3


In [ ]:
output_dir = Path(OUTPUT_DIR)

# JSON
output_data = {
    "metadata": {
        "model_name"      : MODEL_NAME,
        "total_queries"   : len(retrieval_results),
        "total_candidates": len(psc_ids),
        "max_k"           : MAX_K,
        "k_values"        : K_VALUES,
        "index_type"      : "FAISS IndexFlatIP",
    },
    "retrieval_results": {
        query_id: [
            {"candidate_id": cand_id, "similarity": score, "rank": rank}
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in retrieval_results.items()
    }
}

json_path = output_dir / f"retrieval_results_{MODEL_NAME}_top{MAX_K}.json"
with open(json_path, 'w') as f:
    json.dump(output_data, f, indent=2)
print(f"✓ JSON  → {json_path}")

# PKL
pkl_path = output_dir / f"retrieval_results_{MODEL_NAME}_top{MAX_K}.pkl"
with open(pkl_path, 'wb') as f:
    pickle.dump(retrieval_results, f)
print(f"✓ PKL   → {pkl_path}")

## Save reranking-ready pkl (with texts for PhilBERTa CrossEncoder)

In [ ]:
# Format:
# {
#   query_id: {
#       "query_text": str,
#       "candidates": [
#           {"candidate_id": str, "candidate_text": str, "retrieval_score": float, "retrieval_rank": int},
#           ...
#       ]
#   }
# }

rerank_data = {}
missing_texts = 0

for query_id, candidates in tqdm(retrieval_results.items(), desc="Building rerank input"):
    query_text = bdc_text_lookup.get(query_id, "")
    if not query_text:
        missing_texts += 1

    rerank_data[query_id] = {
        "query_text": query_text,
        "candidates": [
            {
                "candidate_id"     : cand_id,
                "candidate_text"   : psc_text_lookup.get(cand_id, ""),
                "retrieval_score"  : score,
                "retrieval_rank"   : rank,
            }
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
    }

if missing_texts:
    print(f"{missing_texts} queries had no text found — check ID_COLUMN / TEXT_COLUMN settings")
else:
    print("All texts resolved")

rerank_pkl_path = output_dir / f"rerank_input_{MODEL_NAME}_top{MAX_K}.pkl"
with open(rerank_pkl_path, 'wb') as f:
    pickle.dump(rerank_data, f)
print(f"Rerank input PKL → {rerank_pkl_path}")

# Quick preview
sample = rerank_data[bdc_ids[0]]
print(f"\nSample query text : {sample['query_text'][:100]}...")
print(f"Top candidate text: {sample['candidates'][0]['candidate_text'][:100]}...")

Building rerank input:   0%|          | 0/19466 [00:00<?, ?it/s]

All texts resolved
Rerank input PKL → /content/retrieval/rerank_input_laberta_vulg_top100.pkl

Sample query text : Epistola ad Rodolphum Asper de scripturae negotio ....
Top candidate text: abbate S. Albani in Anglia ( In Vitis viginti trium abbat. S. Albani ). « Iste, inquit, invalescente...


## Re-rank with PhilBERTa CrossEncoder

In [ ]:
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import CrossEncoder
import torch
import torch.nn.functional as F

RERANKER_MODEL = "julian-schelb/PhilBerta-class-latin-intertext-v1"
RERANK_BATCH_SIZE = 512   
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FINAL_K = 20              # keep after reranking

print(f"Loading re-ranker: {RERANKER_MODEL}")
print(f"Device: {DEVICE}")

reranker = CrossEncoder(
    RERANKER_MODEL,
    device=DEVICE,
    max_length=512,
    tokenizer_kwargs={"truncation": True, "max_length": 512}
)

# Check output shape on a dummy batch to determine softmax vs sigmoid
dummy = reranker.predict([("test", "test")], show_progress_bar=False)
USE_SOFTMAX = len(dummy.shape) > 1 and dummy.shape[1] > 1
print(f"Output shape: {dummy.shape} → using {'softmax' if USE_SOFTMAX else 'sigmoid'}")


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

reranked_results = {}   # query_id → [(candidate_id, rerank_score), ...]

for query_id, data in tqdm(rerank_data.items(), desc="Re-ranking"):
    query_text = data["query_text"]
    cands      = data["candidates"]

    if not cands or not query_text:
        reranked_results[query_id] = []
        continue

    # Guard against empty texts (causes CUDA errors)
    pairs = [
        (query_text if query_text else ".", c["candidate_text"] if c["candidate_text"] else ".")
        for c in cands
    ]

    # Score all pairs
    raw_scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    raw_tensor = torch.tensor(raw_scores)

    # Convert to P(positive) — softmax if 2D output, sigmoid if 1D
    if raw_tensor.ndim == 2:
        probs = F.softmax(raw_tensor, dim=1)[:, 1].numpy()   # positive class via softmax
    else:
        probs = torch.sigmoid(raw_tensor).numpy()            # single logit via sigmoid

    # Sort by probability descending
    ranked = sorted(
        zip([c["candidate_id"] for c in cands], probs),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_results[query_id] = ranked[:FINAL_K]

print(f"  Queries re-ranked : {len(reranked_results):,}")
print(f"  Top-k kept        : {FINAL_K}")

Re-ranking:   0%|          | 0/19466 [00:00<?, ?it/s]

## Save re-ranked results (eval-compatible pkl + JSON)

In [ ]:
RERANKED_MODEL_NAME = f"{MODEL_NAME}_philberta_reranked"

# PKL
reranked_pkl_path = output_dir / f"retrieval_results_{RERANKED_MODEL_NAME}.pkl"
with open(reranked_pkl_path, 'wb') as f:
    pickle.dump(reranked_results, f)
print(f"Re-ranked PKL → {reranked_pkl_path}")

# JSON
reranked_output = {
    "metadata": {
        "model_name"      : RERANKED_MODEL_NAME,
        "retriever"       : MODEL_NAME,
        "reranker"        : RERANKER_MODEL,
        "retrieval_top_k" : MAX_K,
        "reranked_top_k"  : FINAL_K,
        "total_queries"   : len(reranked_results),
        "total_candidates": len(psc_ids),
        "k_values"        : [5, 10, 20],
        "index_type"      : "FAISS IndexFlatIP + PhilBERTa CrossEncoder",
    },
    "retrieval_results": {
        query_id: [
            {"candidate_id": cand_id, "similarity": float(score), "rank": rank}
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in reranked_results.items()
    }
}

reranked_json_path = output_dir / f"retrieval_results_{RERANKED_MODEL_NAME}.json"
with open(reranked_json_path, 'w') as f:
    json.dump(reranked_output, f, indent=2)
print(f"Re-ranked JSON → {reranked_json_path}")